# Testing Sparsity

Our esm embeddings seem to be highly correlated. The popDMS framework is having trouble picking out the real selection coefficients for values under almost any fitness regime

## InterPLM

a paper from stanford using a SAE on top of the PLM embeddings. the SAE was shown to find distinct biological features from the superposition of embedding values from the protein sequences. Can running popDMS on these abstracted features provide better results than simply running them on the highly-correlated emnbedding space? 

In [1]:
test = 1

## PCA's

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import sys

sys.path.insert(0, "/net/dali/home/barton/dhw28/popDMS/esmDMS")
from esmdmsfunctions import (
    get_eigenvector_simulation_results,
    generate_selection, gaussian_selection, zero_selection,
    z_normalize, load_final_df,
)
from scipy.stats import pearsonr

### Configuration — set paths and run parameters here

In [2]:
import os

# ── Paths to analyse (name → directory containing layer0/, layer1/, … subdirs)
comb_path = "/net/dali/home/barton/dhw28/popDMS/esmDMS/data/bg_bf_comb_data"
bg_path   = os.path.expanduser("~/popDMS/esmDMS/data/inference_data/BG505")
bf_path   = os.path.expanduser("~/popDMS/esmDMS/data/inference_results")

PATHS = {
    "comb":  comb_path,
    "BG505": bg_path,
    "BF520": bf_path,
}

# ── Layers to analyze
LAYERS = [12]

# ── Simulation parameters
N_GENS      = 30      # number of generations to simulate
SAVE_EVERY  = 1       # save counts every N generations
FITNESS_FN  = 'exp'   # 'plus1' or 'exp'

# ── Eigenvector / PCA parameters
VARIANCE_EXPLAINED_CUTOFF = 0.95   # keep PCs that together explain this fraction of variance
WEIGHT_BY_INITIAL_FREQ    = False  # weight covariance matrix by pre-selection counts

# ── Selection function (generate_selection, gaussian_selection, or zero_selection)
SEL_FUNC = generate_selection

# ── Inference flags
RUN_INFERENCE    = True
RUN_GAMMA        = False
CALC_ERROR_BARS  = False
INFER_IGNORED    = True

### Run eigenvector simulation

In [ ]:
all_results = {}   # path_name -> (all_layer_fits, all_sel_coeffs, detailed_results, gamma_results, all_gen_counts, eig_info)

for path_name, path in PATHS.items():
    print(f"\n{'='*60}")
    print(f"Dataset: {path_name}  |  {path}")
    print('='*60)

    results = get_eigenvector_simulation_results(
        n_gens=N_GENS,
        embedding_df_path=path,
        sel_func=SEL_FUNC,
        inference=RUN_INFERENCE,
        gamma_analysis=RUN_GAMMA,
        fitness=FITNESS_FN,
        save_every=SAVE_EVERY,
        layers=LAYERS,
        variance_explained_cutoff=VARIANCE_EXPLAINED_CUTOFF,
        weight_by_initial_freq=WEIGHT_BY_INITIAL_FREQ,
        calc_error_bars=CALC_ERROR_BARS,
        infer_ignored_dims=INFER_IGNORED,
    )
    all_results[path_name] = results

print("\nDone. Datasets completed:", list(all_results.keys()))

### Scree plot — variance explained by each eigenvector

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')

for path_name, results in all_results.items():
    all_layer_fits, all_sel_coeffs, detailed_results, gamma_results, all_gen_counts, eig_info = results

    for layer in LAYERS:
        info = eig_info[layer]
        eigenvalues = np.maximum(info['eigenvalues'], 0.0)
        cum_var = np.cumsum(eigenvalues) / eigenvalues.sum()
        n_keep = info['n_components']

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        axes[0].bar(range(1, len(eigenvalues) + 1), eigenvalues / eigenvalues.sum(), color='steelblue')
        axes[0].axvline(n_keep + 0.5, color='red', linestyle='--', label=f'Cutoff @ PC {n_keep}')
        axes[0].set_xlabel('Principal Component')
        axes[0].set_ylabel('Fraction of Variance Explained')
        axes[0].set_title(f'[{path_name}]  Layer {layer} — Scree Plot')
        axes[0].legend()

        axes[1].plot(range(1, len(cum_var) + 1), cum_var, marker='o', markersize=3, color='steelblue')
        axes[1].axhline(VARIANCE_EXPLAINED_CUTOFF, color='orange', linestyle='--',
                        label=f'Cutoff = {VARIANCE_EXPLAINED_CUTOFF:.0%}')
        axes[1].axvline(n_keep + 0.5, color='red', linestyle='--', label=f'PC {n_keep} retained')
        axes[1].set_xlabel('Number of Principal Components')
        axes[1].set_ylabel('Cumulative Variance Explained')
        axes[1].set_title(f'[{path_name}]  Layer {layer} — Cumulative Variance')
        axes[1].legend()

        plt.tight_layout()
        plt.show()
        print(f"[{path_name}]  Layer {layer}: {n_keep} PCs retained, "
              f"{info['variance_explained']*100:.1f}% variance explained")

### True vs inferred selection coefficients (in eigenvector space)

In [ ]:
MAX_COLS = 3

for path_name, results in all_results.items():
    all_layer_fits, all_sel_coeffs, detailed_results, gamma_results, all_gen_counts, eig_info = results

    for layer in LAYERS:
        if layer not in detailed_results:
            print(f"[{path_name}]  Layer {layer}: no inference results (run with inference=True)")
            continue

        true_s       = all_sel_coeffs[layer]
        s_reps       = detailed_results[layer][0]
        n_reps       = len(s_reps)
        n_components = eig_info[layer]['n_components']

        n_cols = min(n_reps, MAX_COLS)
        n_rows = int(np.ceil(n_reps / n_cols))

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows), squeeze=False)

        for rep in range(n_reps):
            row, col = divmod(rep, n_cols)
            ax = axes[row][col]

            true_norm     = z_normalize(true_s)
            inferred_norm = z_normalize(s_reps[rep])

            ax.scatter(true_norm, inferred_norm, alpha=0.7, edgecolors='k', linewidths=0.3)
            lim = max(np.abs(true_norm).max(), np.abs(inferred_norm).max()) + 0.5
            ax.plot([-lim, lim], [-lim, lim], 'r--')
            ax.set_xlabel('True s (normalized)')
            ax.set_ylabel('Inferred s (normalized)')
            ax.set_title(f'Rep {rep + 1}')
            ax.axis('equal')

            corr, pval = pearsonr(true_norm, inferred_norm)
            ax.annotate(f'r = {corr:.3f}\np = {pval:.2e}',
                        xy=(0.05, 0.95), xycoords='axes fraction',
                        ha='left', va='top', fontsize=10,
                        bbox=dict(boxstyle='round', fc='white', alpha=0.8))

        for idx in range(n_reps, n_rows * n_cols):
            row, col = divmod(idx, n_cols)
            axes[row][col].set_visible(False)

        fig.suptitle(f'[{path_name}]  Layer {layer} — True vs Inferred s  ({n_components} eigenvectors)', fontsize=13)
        plt.tight_layout()
        plt.show()

### Cross-replicate consistency of inferred s

In [ ]:
MAX_COLS = 3

for path_name, results in all_results.items():
    all_layer_fits, all_sel_coeffs, detailed_results, gamma_results, all_gen_counts, eig_info = results

    for layer in LAYERS:
        if layer not in detailed_results:
            print(f"[{path_name}]  Layer {layer}: no inference results")
            continue

        s_reps     = detailed_results[layer][0]
        n_reps     = len(s_reps)
        rep_pairs  = [(i, j) for i in range(n_reps) for j in range(i + 1, n_reps)]
        n_plots    = len(rep_pairs)

        n_cols = min(n_plots, MAX_COLS)
        n_rows = int(np.ceil(n_plots / n_cols))

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows), squeeze=False)

        for idx, (ri, rj) in enumerate(rep_pairs):
            row, col = divmod(idx, n_cols)
            ax = axes[row][col]

            si = z_normalize(s_reps[ri])
            sj = z_normalize(s_reps[rj])
            ax.scatter(si, sj, alpha=0.6, edgecolors='k', linewidths=0.3)
            lim = max(np.abs(si).max(), np.abs(sj).max()) + 0.5
            ax.plot([-lim, lim], [-lim, lim], 'r--')
            ax.set_xlabel(f'Rep {ri + 1} s (normalized)')
            ax.set_ylabel(f'Rep {rj + 1} s (normalized)')
            corr, pval = pearsonr(si, sj)
            ax.set_title(f'r = {corr:.3f}  (p = {pval:.2e})')
            ax.axis('equal')

        for idx in range(n_plots, n_rows * n_cols):
            row, col = divmod(idx, n_cols)
            axes[row][col].set_visible(False)

        fig.suptitle(f'[{path_name}]  Layer {layer} — Cross-replicate consistency  ({eig_info[layer]["n_components"]} eigenvectors)', fontsize=13)
        plt.tight_layout()
        plt.show()